In [ ]:
def correction_function(x):
    return x

In [ ]:
import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../../../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../data_generation_pipeline"))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from spectra_stitching import load_forest_spectra

In [ ]:
test_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/reference/lya_forest_spectra/forest_spectra.hdf5"
test_range = (3800, 4500)
n_specs = 1
redshifts_to_use = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]

#wave_master, flux_master, usage_records = create_forest_spectra(test_path, test_range, redshifts_to_use, correction_function, n_specs)
wave_master, flux_master, usage_records = load_forest_spectra(test_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import h5py
from matplotlib.ticker import FixedLocator
import colorsys


def get_redshift_color_map(usage_records_for_one_spectrum, base_cmap=plt.cm.Set1):
    """
    Assign each unique redshift a base color, and each repeat within that
    redshift a lighter/darker shade of the same hue.

    Returns a list of colors, same length and order as usage_records_for_one_spectrum.
    """
    # unique redshifts in order of first appearance
    unique_zs = []
    for rec in usage_records_for_one_spectrum:
        if rec["sim_redshift"] not in unique_zs:
            unique_zs.append(rec["sim_redshift"])

    n_unique = len(unique_zs)
    base_colors = base_cmap(np.linspace(0, 1, n_unique)) if n_unique <= 10 else plt.cm.tab20(np.linspace(0, 1, n_unique))
    base_color_map = {z: base_colors[i][:3] for i, z in enumerate(unique_zs)}  # drop alpha

    # count how many repeats each redshift has, to know how to space shades
    counts = {}
    for rec in usage_records_for_one_spectrum:
        counts[rec["sim_redshift"]] = counts.get(rec["sim_redshift"], 0) + 1

    seen_so_far = {z: 0 for z in unique_zs}
    colors = []
    for rec in usage_records_for_one_spectrum:
        z = rec["sim_redshift"]
        r, g, b = base_color_map[z]
        h, l, s = colorsys.rgb_to_hls(r, g, b)

        n_repeats_this_z = counts[z]
        idx_within_z = seen_so_far[z]
        seen_so_far[z] += 1

        if n_repeats_this_z == 1:
            lightness = l  # no variation needed
        else:
            # spread lightness values around the base, e.g. from -0.2 to +0.2
            spread = np.linspace(0, 0.2, n_repeats_this_z)
            lightness = np.clip(l + spread[idx_within_z], 0.15, 0.85)

        r2, g2, b2 = colorsys.hls_to_rgb(h, lightness, s)
        colors.append((r2, g2, b2))

    return colors


Line_wavelength_restframe = 1215.67

n_segments = len(usage_records[0])

fig = plt.figure(figsize=(2 * n_segments, 8))
gs = gridspec.GridSpec(2, n_segments, height_ratios=[1, 1.3], hspace=0.05, wspace=0.05)

ax_main = fig.add_subplot(gs[1, :])

colors = get_redshift_color_map(usage_records[0])
#colors = plt.cm.tab10(np.linspace(0, 1, n_segments)) if n_segments <= 10 else plt.cm.tab20(np.linspace(0, 1, n_segments))

for i, spec_info in enumerate(usage_records[0]):
    spec_path = spec_info["spec_path"]
    spec_idx = spec_info["spec_idx"]
    spec_z = spec_info["sim_redshift"]
    spec_dz = spec_info["redshift_width"]
    long_spec_wave_start = spec_info["left_wavelength"]

    short_spec_wave_start = (spec_z + 1) * Line_wavelength_restframe
    short_spec_wave_end = (spec_z + spec_dz + 1) * Line_wavelength_restframe

    shift = long_spec_wave_start - short_spec_wave_start

    with h5py.File(spec_path, "r") as f:
        wavelengths = f["wave"][:].copy()
        taus = f["tau_HI_1215"][spec_idx]

    flux = np.exp(-taus)

    color = colors[i]

    # small individual panel on top
    ax_small = fig.add_subplot(gs[0, i])
    if i == 0:
        ax_small.set_ylabel("Relative Flux")
    ax_small.plot(wavelengths, flux, color=color)
    ax_small.fill_between(wavelengths, flux, 1, color=color, alpha=0.4)
    ax_small.set_xlim(short_spec_wave_start, short_spec_wave_end)
    ax_small.set_ylim(0, 1.05)
    ax_small.set_xticks([])
    if i > 0:
        ax_small.set_yticks([])
    ax_small.text(0.05, 0.05, f"z={spec_z:.1f}", transform=ax_small.transAxes)

    wavelengths = wavelengths + shift

    # same segment, filled, in the main combined panel
    ax_main.plot(wavelengths, flux, color=color, alpha=0.6, zorder=1)
    ax_main.fill_between(wavelengths, flux, 1, color=color, alpha=0.4, zorder=1)

# black patched spectrum drawn last/on top
ax_main.plot(wave_master, flux_master[0], color="black", zorder=2, label="patched spectrum")

ax_main.set_xlim(*test_range)
ax_main.set_ylim(-0.15, 1.05)
ax_main.set_xlabel("Wavelength [Å]")
ax_main.set_ylabel("Relative Flux")
ax_main.legend(loc="lower left", bbox_to_anchor=(0.0, 0.08))

# --- secondary redshift axis, pushed below the wavelength axis ---
def wave_to_z(wave):
    return wave / Line_wavelength_restframe - 1

def z_to_wave(z):
    return (z + 1) * Line_wavelength_restframe

z_min, z_max = wave_to_z(test_range[0]), wave_to_z(test_range[1])
candidate_zs = np.arange(np.ceil(z_min * 10) / 10, z_max, 0.1)  # every 0.1 in z, adjust as needed

secax = ax_main.secondary_xaxis(0, functions=(wave_to_z, z_to_wave))

# keep the normal formatter/locator for all ticks (with your z-based filtering removed if you want ALL ticks back)
secax.xaxis.set_major_locator(FixedLocator(candidate_zs))
secax.xaxis.set_major_formatter(lambda x, pos: f"z={x:.1f}")

# force a draw so tick label positions are computed
fig.canvas.draw()

edge_margin_wave = 15  # Å; how close counts as "near the edge"
shift_points = 12      # how far inward to nudge the label, in points

for tick, z_val in zip(secax.xaxis.get_major_ticks(), candidate_zs):
    label = tick.label1
    wave = z_to_wave(z_val)

    if (wave - test_range[0]) <= edge_margin_wave:
        label.set_horizontalalignment("left")
        label.set_transform(label.get_transform() + plt.matplotlib.transforms.ScaledTranslation(
            shift_points / 72, 0, fig.dpi_scale_trans))
    elif (test_range[1] - wave) <= edge_margin_wave:
        label.set_horizontalalignment("right")
        label.set_transform(label.get_transform() + plt.matplotlib.transforms.ScaledTranslation(
            -shift_points / 72, 0, fig.dpi_scale_trans))


# ticks pointing inward, labels pulled up into the plot via negative pad
secax.tick_params(axis="x", direction="in", width=2.0, length=6, pad=-30)

# hide the secondary axis's own spine/line so it doesn't duplicate the main bottom spine
secax.spines["bottom"].set_visible(False)

fig.subplots_adjust(bottom=0.2)

fig.suptitle("")

plt.savefig("plots/augmented_spectra.pdf", format="pdf")
plt.show()